In [14]:
import xarray as xr
import numpy as np
import at3d

In [15]:
def get_MISR_mixture_optical_properties(band, mixture, aod, rte_grid, acp_file, legendre_order=12, g_tol=1e-3):
    # Read in MISR aerosol data
    acp_all = xr.load_dataset(acp_file)
    acp = acp_all.sel(mixture=mixture, band=band, method='nearest').copy(deep=True)

    # Perform Legendre function expansion
    x = np.cos(acp.theta.data)
    y = acp.phase.data
    legendre_coeffs_P11, leg_quality = np.polynomial.legendre.legfit(x, y, legendre_order, full=True)

    legendre_coeffs_P11 /= legendre_coeffs_P11[0] # normalization

    assert np.abs(legendre_coeffs_P11[1]/3 - acp.g.data) < g_tol, 'Asymmetry tolerance not met. Expected g = {:f} and got g = {:f}'.format(acp.g.data, legendre_coeffs_P11[1]/3)

    StokesIndex = 6
    legendreIndex = legendre_order + 1
    PhaseFunctionIndex = 1

    phase_function_table = np.zeros((StokesIndex, legendreIndex, PhaseFunctionIndex))
    phase_function_table[0, :, 0] = legendre_coeffs_P11
    phase_function_indicies = np.zeros((1, rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z']))

    # Plug aerosol profile into rte_grid
    rte_grid = aerosol_vertical_distribution(aod, mixture, rte_grid)

    extinction = np.tile(rte_grid.z_ext.data, (rte_grid.dims['x'], rte_grid.dims['y'], 1))
    ssalb = acp.ssalb.data * np.ones((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z']))

    # Generate optical properties
    aerosol_optical_properties = at3d.medium.make_optical_properties(
        grid=rte_grid,
        extinction=extinction,
        ssalb=ssalb,
        phase_function_indices=phase_function_indicies,
        phase_function_table=phase_function_table,
        phase_function_weights=None
    )
    aerosol_optical_properties = aerosol_optical_properties.assign_attrs(
        mixture=acp.mixture.data,
        band=acp.band.data,
        leg_residuals=leg_quality[0]
    )
    return aerosol_optical_properties

def aerosol_vertical_distribution(aod, mixture, rte_grid):
    z = rte_grid.z.data
    if(mixture <= 50):
        H = 2
        B0 = -aod / (H*(np.exp(-10/H) - 1))
        B = B0 * np.exp(-z/H)
        B[z > 10] = 0
    else:
        H = 10
        B0 = -aod / (H*(np.exp(-3/H) - 1))
        B = B0 * np.exp(-(z-3)/H)
        B[z < 3] = 0
        B[z > 6] = 0
    rte_grid['z_ext'] = (['z'], B)
    rte_grid_minus1 = rte_grid.isel(z = slice(0, -1)).copy(deep=True)
    tau_minus1 = rte_grid_minus1.z_ext.integrate('z').data
    h = rte_grid.z.data[-1] - rte_grid.z.data[-2]
    xmax = rte_grid_minus1.z_ext.data[-1]
    del_tau = aod - tau_minus1
    compensation_extinction = ((2*del_tau)/h) - xmax
    B[-1] = compensation_extinction
    rte_grid['z_ext'] = (['z'], B)
    return rte_grid

In [16]:
rte_grid = at3d.grid.make_grid(0.275, 65, 0.275, 65, np.arange(0, 1.3, 0.1))
aerosol_optical_properties = get_MISR_mixture_optical_properties(
    band=0.67,
    mixture=54,
    aod=0.2,
    rte_grid=rte_grid,
    acp_file='../data/MISR/MISR_AM1_ACP_MIXTURE_F06_0022.nc',
    legendre_order=20,
    g_tol=1e-2
)
aerosol_optical_properties

<xarray.Dataset>
Dimensions:        (x: 65, y: 65, z: 13, num_micro: 1, stokes_index: 6,
                    legendre_index: 21, table_index: 1)
Coordinates:
  * x              (x) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * y              (y) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * z              (z) float64 0.0 0.1 0.2 0.3 0.4 0.5 ... 0.8 0.9 1.0 1.1 1.2
    table_index    (num_micro, x, y, z) int32 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1
Dimensions without coordinates: num_micro, stokes_index, legendre_index
Data variables:
    delx           float64 0.275
    dely           float64 0.275
    z_ext          (z) float64 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 4.0
    ssalb          (x, y, z) float64 0.9972 0.9972 0.9972 ... 0.9972 0.9972
    extinction     (x, y, z) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 4.0
    phase_weights  (num_micro, x, y, z) float64 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
    legcoef        (stokes_index, legendre_index, table_index) float64 1.0 .....
Attributes:
    mixture:        54
    band:           0.67175
    leg_residuals:  [1650.3723]